# 05 - Cross-Platform Validation

Explore Golden Set results from `data/results/cross_platform/`.  
Golden Set = proteins that are DNB core on BOTH SomaScan and Olink platforms.  
Tier 1 = Golden Set + CSD support (strongest evidence).  
**This notebook reads pre-computed results only.**

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

%matplotlib inline
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120

In [ ]:
with open('../config/config.yaml', 'r') as f:
    config = yaml.safe_load(f)

results_dir = os.path.join('..', config['paths']['results'], 'cross_platform')
print('Cross-platform results directory:', results_dir)
print('Files available:', os.listdir(results_dir) if os.path.exists(results_dir) else 'DIRECTORY NOT FOUND')

## Golden Set Overview

Proteins identified as DNB core on both SomaScan and Olink platforms.
These represent the most robust biomarker candidates — their signal
survives orthogonal measurement technologies.

In [ ]:
# Load Golden Set
golden_path = os.path.join(results_dir, 'golden_set.csv')
if os.path.exists(golden_path):
    golden = pd.read_csv(golden_path)
    print(f'Golden Set proteins: {len(golden)}')
    print(f'Tier 1 (Golden Set + CSD): {(golden["tier"] == "Tier_1").sum() if "tier" in golden.columns else "N/A"}')
    print(f'Tier 2 (Golden Set only): {(golden["tier"] == "Tier_2").sum() if "tier" in golden.columns else "N/A"}')
    print()
    print(golden.to_string(index=False))
else:
    print('Golden Set not found. Run: python pipelines/run_full_pipeline.py --stage cross_platform')

## Platform Concordance

Spearman correlation of DNB frequency between SomaScan and Olink
for proteins measurable on both platforms.

In [ ]:
# Load concordance results
conc_path = os.path.join(results_dir, 'concordance_results.csv')
if os.path.exists(conc_path):
    concordance = pd.read_csv(conc_path)
    print(f'Proteins with concordance data: {len(concordance)}')
    print(f'Median Spearman rho: {concordance["spearman_rho"].median():.3f}')
    print(f'Proteins with rho > 0.5: {(concordance["spearman_rho"] > 0.5).sum()}')
    
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.hist(concordance['spearman_rho'], bins=30, color='steelblue', edgecolor='white')
    ax.axvline(concordance['spearman_rho'].median(), color='firebrick', linestyle='--',
               label=f'Median = {concordance["spearman_rho"].median():.3f}')
    ax.set_xlabel('Spearman rho')
    ax.set_ylabel('Count')
    ax.set_title('Cross-Platform Concordance Distribution')
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print('Concordance results not found.')

## Positive Control Check

Established neurodegeneration biomarkers (GFAP, NEFL, CLU, APOE) should
show concordant signals on both platforms. If they don't, the cross-platform
comparison is unreliable.

In [ ]:
# Load concordance report
report_path = os.path.join(results_dir, 'concordance_report.txt')
if os.path.exists(report_path):
    with open(report_path) as f:
        print(f.read())
else:
    print('Concordance report not found.')

## Golden Set Frequency Scatter

For Golden Set proteins, compare DNB frequency on SomaScan (x-axis)
vs. Olink (y-axis). Points above the diagonal are more frequent on Olink.

In [ ]:
if os.path.exists(golden_path):
    golden = pd.read_csv(golden_path)
    if 'frequency_somascan' in golden.columns and 'frequency_olink' in golden.columns:
        gs = golden[golden['is_golden_set'] == True] if 'is_golden_set' in golden.columns else golden
        
        fig, ax = plt.subplots(figsize=(7, 7))
        ax.scatter(gs['frequency_somascan'], gs['frequency_olink'],
                   c='darkorange', s=40, alpha=0.7, edgecolors='black', linewidths=0.5)
        
        lims = [0, max(gs['frequency_somascan'].max(), gs['frequency_olink'].max()) + 0.05]
        ax.plot(lims, lims, 'k--', alpha=0.3, label='y=x')
        ax.set_xlabel('DNB Frequency (SomaScan)')
        ax.set_ylabel('DNB Frequency (Olink)')
        ax.set_title('Golden Set: Cross-Platform DNB Frequency')
        ax.legend()
        plt.tight_layout()
        plt.show()
        
        rho, p = stats.spearmanr(gs['frequency_somascan'], gs['frequency_olink'])
        print(f'Spearman rho = {rho:.3f}, p = {p:.2e}')
    else:
        print('Frequency columns not found in Golden Set data.')
else:
    print('Golden Set not found.')